# Query Strings

A **query string** is the optional portion of a URL that starts after a `?` character and contains key-value pairs separated by `&`. In API design, query strings are primarily used in `GET` requests to control what data the server returns.

---

## Structure

```
https://example.com/api/products?category=shoes&size=10&sort=price
                                ↑        ↑     ↑
                                |        |     └── & connects further parameters
                                |        └──────── first key-value pair
                                └───────────────── ? separates path from parameters
```

- **`?`** — separates the base URL path from the parameters.
- **`key=value`** — a single parameter.
- **`&`** — connects multiple parameters together.

---

## Core Use Cases in APIs

| Use case | Purpose | Example |
| --- | --- | --- |
| **Filtering** | Narrowing results to specific criteria | `?status=active` |
| **Pagination** | Dividing large datasets into chunks | `?page=2&limit=50` |
| **Sorting** | Changing the order of returned items | `?sort_by=created_at&order=asc` |
| **Searching** | Passing keywords or partial strings | `?q=wireless+headphones` |

---

## Working with Query Strings Programmatically

Most languages provide built-in tools to parse and safely format query strings, ensuring proper **percent-encoding** of special characters. Building them by hand with string concatenation is where injection and encoding bugs come from.

### 1. JavaScript (browser)

Use the native `URLSearchParams` API.

```javascript
// Constructing a query string
const params = new URLSearchParams({
  search: "laptop core i7",
  min_price: "500"
});

console.log(params.toString());
// Outputs: search=laptop+core+i7&min_price=500

// Parsing a query string from the current URL
const urlParams = new URLSearchParams(window.location.search);
const category = urlParams.get('category');
```

Attaching it to a `fetch` call:

```javascript
const params = new URLSearchParams({ page: 2, limit: 50 });
const response = await fetch(`https://example.com/api/products?${params}`);
```

> Note the spaces became `+`, and any `&`, `=`, or `#` inside a value would be percent-encoded automatically. This is the reason to use the API rather than template-stringing values in yourself.

### 2. Node.js

```javascript
import querystring from 'node:querystring';

// Parsing a raw string into an object
const rawString = "type=admin&session=xyz123";
const parsed = querystring.parse(rawString);

console.log(parsed.type); // Outputs: admin
```

`URLSearchParams` is also available globally in Node and is generally the preferred modern option.

### 3. Python

```python
from urllib.parse import urlencode, parse_qs

params = {'results': 20, 'tags': 'electronics'}
query_string = urlencode(params)
print(query_string)  # Outputs: results=20&tags=electronics

# Parsing back
parse_qs('results=20&tags=electronics')
# {'results': ['20'], 'tags': ['electronics']}
```

With `requests`, pass a dict to `params` and the encoding is handled for you:

```python
requests.get('https://example.com/api/products', params={'brand': 'apple'})
```

---

## Query Parameters vs. Path Parameters

| Feature | Query parameters (`?key=value`) | Path parameters (`/api/items/:id`) |
| --- | --- | --- |
| **Primary purpose** | Filtering, sorting, pagination | Pinpointing a specific resource instance |
| **Requirement** | Almost always optional | Strictly required by the URL structure |
| **Example** | `/products?brand=apple` | `/products/10293` |

**Rule of thumb:** if removing it still gives you a valid request that returns something sensible, it's a query parameter. If removing it makes the URL meaningless, it belongs in the path.

---

## Gotchas

- **Values are always strings.** `?limit=50` arrives as `"50"`, not `50`. Cast on the server.
- **Repeated keys are legal.** `?tag=a&tag=b` is valid; `URLSearchParams.get()` returns only the first, `getAll()` returns both. Python's `parse_qs` returns lists for this reason.
- **Never put secrets in a query string.** They land in browser history, server access logs, and `Referer` headers. Credentials and tokens go in headers or the request body.